In [ ]:
"""
Objetivo: Executar o pipeline completo de Machine Learning do estudo. 
Inclui Aprendizado Não Supervisionado (K-Means para detecção de regimes macroeconômicos) 
e Aprendizado Supervisionado (XGBoost com SHAP Values para interpretabilidade preditiva).
"""

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import xgboost as xgb
import shap

warnings.filterwarnings('ignore')

# ==========================================
# PARÂMETROS GERAIS E CARREGAMENTO
# ==========================================
ARQUIVO_BASE = 'base_macro_fred_limpa.csv'
INICIO, FIM = '2005-01-01', '2026-04-01'

print("[*] Carregando a base de dados limpa...")
df_completo = pd.read_csv(ARQUIVO_BASE, index_col=0, parse_dates=True)
df_model = df_completo.loc[INICIO:FIM].copy()

# ==============================================================================
# BLOCO A: APRENDIZADO NÃO SUPERVISIONADO (K-MEANS E LIMITES DE REGIME)
# ==============================================================================
def executar_kmeans(df: pd.DataFrame):
    print("\n--- BLOCO A: K-MEANS E DETECÇÃO DE REGIMES ---")
    features_kmeans = ['esforco_pagamento', 'desemprego']
    df_cluster = df[features_kmeans].dropna()
    
    # Padronização
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_cluster)
    
    # 1. Otimização Dinâmica (k=2 a 8) via Silhouette Score
    melhor_k, melhor_silhueta = 2, -1
    inercia_valores = []
    k_range = range(2, 9)
    
    for k in k_range:
        kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans_temp.fit_predict(X_scaled)
        silhueta = silhouette_score(X_scaled, labels)
        inercia_valores.append(kmeans_temp.inertia_)
        
        if silhueta > melhor_silhueta:
            melhor_silhueta = silhueta
            melhor_k = k
            
    print(f"[*] Otimização concluída. K ótimo = {melhor_k} (Silhouette: {melhor_silhueta:.3f})")
    
    # 2. Estimação Final
    kmeans_final = KMeans(n_clusters=melhor_k, random_state=42, n_init=10)
    df_cluster['Cluster'] = kmeans_final.fit_predict(X_scaled)
    centroides = scaler.inverse_transform(kmeans_final.cluster_centers_)
    
    # Nomenclatura baseada na posição do centróide
    nomes_clusters = {}
    for c in range(melhor_k):
        esf, des = centroides[c, 0], centroides[c, 1]
        if des < 7.5:
            nomes_clusters[c] = 'Baixo Esforço e Baixo Desemprego' if esf < 12.5 else 'Alto Esforço e Baixo Desemprego'
        else:
            nomes_clusters[c] = 'Alto Esforço e Alto Desemprego'
            
    # 3. Exportação das Estatísticas
    limites_regimes = df_cluster.groupby('Cluster').agg({
        'esforco_pagamento': ['min', 'max'], 'desemprego': ['min', 'max']
    }).round(2)
    limites_regimes.index = limites_regimes.index.map(lambda x: nomes_clusters.get(x, f'Cluster {x}'))
    
    csv_filename = '04A_estatisticas_regimes_kmeans.csv'
    limites_regimes.to_csv(csv_filename, encoding='utf-8')
    print(f"[*] Estatísticas exportadas para '{csv_filename}'")
    
    # 4. Gráfico de Dispersão K-Means
    cores = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    fig, ax = plt.subplots(figsize=(9, 6))
    
    for c in range(melhor_k):
        subset = df_cluster[df_cluster['Cluster'] == c]
        ax.scatter(subset['esforco_pagamento'], subset['desemprego'], color=cores[c], 
                   alpha=0.75, s=60, edgecolors='k', linewidths=0.4, label=nomes_clusters[c])
        
    ax.scatter(centroides[:, 0], centroides[:, 1], color='black', marker='X', s=180, zorder=5, label='Centróides')
    
    ax.set_xlabel('Esforço de Pagamento das Famílias (%)', fontsize=10, fontweight='bold')
    ax.set_ylabel('Taxa de Desemprego (%)', fontsize=10, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.legend(frameon=False, loc='best', fontsize=9)
    plt.tight_layout()
    plt.savefig('04A_grafico_kmeans.png', dpi=300, bbox_inches='tight')
    plt.close()

# ==============================================================================
# BLOCO B: APRENDIZADO SUPERVISIONADO (XGBOOST E SHAP VALUES)
# ==============================================================================
def executar_xgboost_shap(df: pd.DataFrame):
    print("\n--- BLOCO B: XGBOOST E SHAP VALUES ---")
    ALVO = 'inad_cartoes'
    FEATURES = ['desemprego', 'fed_funds', 'spread_10y3m', 'esforco_pagamento', 'sloos_aperto']
    LAG_VENCEDOR = 1
    PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)

    # Criação do Dataset com features defasadas
    X = pd.DataFrame(index=df.index)
    for col in FEATURES:
        X[f'{col}_lag{LAG_VENCEDOR}'] = df[col].shift(LAG_VENCEDOR)
        
    linhas_validas = pd.concat([X, df[ALVO]], axis=1).dropna().index
    X_final, y_final = X.loc[linhas_validas], df.loc[linhas_validas, ALVO]
    
    print(f"[*] Treinando XGBoost (Lag {LAG_VENCEDOR})...")
    modelo_final = xgb.XGBRegressor(**PARAMS).fit(X_final, y_final)
    
    print("[*] Extraindo valores SHAP...")
    explainer = shap.TreeExplainer(modelo_final)
    shap_values = explainer.shap_values(X_final)
    
    # 1. Gráfico de Importância Global (Mean Absolute SHAP)
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    nomes_limpos = [col.replace(f'_lag{LAG_VENCEDOR}', '').replace('_', ' ').title() for col in X_final.columns]
    
    df_plot = pd.DataFrame({'Feature': nomes_limpos, 'Importance': mean_abs_shap}).sort_values('Importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(12, 7))
    barras = ax.barh(df_plot['Feature'], df_plot['Importance'], color='#d62728', height=0.6)
    
    for barra in barras:
        width = barra.get_width()
        ax.text(width + 0.005, barra.get_y() + barra.get_height()/2, f'{width:.3f}', va='center', fontweight='bold', fontsize=10)
        
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.grid(True, linestyle='--', alpha=0.4, color='gray')
    ax.set_xlabel('Impacto Médio no Risco de Inadimplência (|SHAP Value|)', fontsize=12, fontweight='bold', labelpad=15)
    
    ax.legend(handles=[Patch(facecolor='#d62728', label='Magnitude Global de Importância')], loc='lower right', frameon=False)
    plt.tight_layout()
    plt.savefig('04B_shap_painel_executivo.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Mapeamento de Crises Históricas na Variável Principal
    var_ruptura = f'esforco_pagamento_lag{LAG_VENCEDOR}'
    idx_esforco = list(X_final.columns).index(var_ruptura)
    
    plt.figure(figsize=(10, 7))
    plt.scatter(X_final[var_ruptura], shap_values[:, idx_esforco], color='#1f77b4', alpha=0.7, s=50, label='Trimestres Históricos')
    
    marcos = {
        '2008-10-01': 'Crise do Subprime (2008)',
        '2020-04-01': 'Choque Pandêmico (2020)',
        '2022-07-01': 'Crise Inflacionária (2022)'
    }
    
    for data_str, rotulo in marcos.items():
        data_ts = pd.to_datetime(data_str)
        if data_ts in X_final.index:
            pos_loc = np.where(X_final.index == data_ts)[0][0]
            x_val, y_val = float(X_final.loc[data_ts, var_ruptura]), float(shap_values[pos_loc, idx_esforco])
            plt.scatter(x_val, y_val, color='red', s=100, zorder=5)
            plt.annotate(rotulo, (x_val, y_val), textcoords="offset points", xytext=(15, -5), fontweight='bold', fontsize=9,
                         bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.5), arrowprops=dict(arrowstyle="->", connectionstyle="arc3,rad=0"))

    plt.axhline(0, color='red', linestyle='--', alpha=0.5)
    plt.xlabel('Esforço de Pagamento das Famílias no Trimestre Anterior (%)', fontsize=11, fontweight='bold')
    plt.ylabel('Impacto SHAP (Força propulsora)', fontsize=11, fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig('04B_shap_ruptura_com_crises.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("[*] Gráficos de interpretabilidade SHAP exportados com sucesso.")

if __name__ == "__main__":
    executar_kmeans(df_model)
    executar_xgboost_shap(df_model)
    print("\n[SUCESSO] Pipeline completo de Machine Learning (Apêndice) finalizado.")